# Basic Template Exploration: MLflow Tracking & Best Practices

This notebook explores the Bike Sharing dataset using the project library, while demonstrating how to incorporate MLflow tracking for reproducible research and following best practices from the NextGen workshops.

## 📦 1. Install and Import Dependencies
First, we ensure our environment is set up. If running in Colab, we clone the repository to access our local package.


In [ ]:
# Colab Environment Setup
import sys
from pathlib import Path

# If running in Google Colab, clone the repo and add src to path
if 'google.colab' in sys.modules:
    !git clone -b template/basic https://github.com/acidsafari/nextgen2026-coding-bootcamp.git
    sys.path.append('/content/nextgen2026-coding-bootcamp/src')
    print("Running in Colab: Repository cloned and src added to path.")
else:
    print("Running locally: Ensure the project is installed in editable mode (uv sync).")


In [ ]:
import mlflow
import mlflow.sklearn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error

# Import our project logic
from nextgen2026_coding_bootcamp.config import load_full_config
from nextgen2026_coding_bootcamp.runtime import RunContext
from nextgen2026_coding_bootcamp.data import _to_rng
from nextgen2026_coding_bootcamp.steps.fetch import run_fetch
from nextgen2026_coding_bootcamp.steps.prepare import run_prepare
from nextgen2026_coding_bootcamp.steps.analyze import run_analyze
from nextgen2026_coding_bootcamp.steps.report import run_report

print("Dependencies imported.")


## 🧪 2. Initialize MLflow Experiment
We set up MLflow to track our analysis runs.


In [ ]:
# Configure MLflow
mlflow.set_tracking_uri("sqlite:///mlflow.db")
experiment_name = "Bike-Sharing-Analysis"
mlflow.set_experiment(experiment_name)

print(f"Experiment set to: {experiment_name}")


## 📊 3. Track Hyperparameters and Metrics
We run the full workflow and log the results to MLflow. We also demonstrate the use of the `_to_rng` helper for reproducibility.


The Mean Squared Error is defined as:
$$MSE = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$


In [ ]:
# Load configuration
cfg = load_full_config(profile="base")

# Demonstrate reproducibility with _to_rng
rng = _to_rng(seed=42)
random_data = rng.standard_normal(5)
print(f"Reproducible random data: {random_data}")

# Start MLflow run
with mlflow.start_run(run_name="Full Workflow Run"):
    # Log parameters from config
    mlflow.log_param("high_demand_quantile", cfg.analysis.high_demand_quantile)
    
    # Run Stages
    fetch_out = run_fetch(cfg)
    prepare_out = run_prepare(cfg)
    analyze_out = run_analyze(cfg)
    
    # Log metrics (manually for demonstration)
    # In a real scenario, these would come from analyze_out
    mlflow.log_metric("rows_in", analyze_out.get("rows_in", 0))
    mlflow.log_metric("threshold", analyze_out.get("high_demand_threshold", 0))
    
    print("Workflow stages completed and logged to MLflow.")


## 🖼️ 4. Log Model Artifacts and Figures
We capture the plots generated by the report stage and log them as artifacts. We use the updated `create_demand_plots` pattern to display the figures directly in the notebook.


In [ ]:
from nextgen2026_coding_bootcamp.steps.report import create_demand_plots

# Define directories
analyze_dir = Path(cfg.paths.results_dir)
output_dir = Path(cfg.paths.results_dir) / "notebook_report"

# Generate plots and get figures
report_out = create_demand_plots(
    hourly_profile_csv=analyze_dir / "hourly_profile.csv",
    high_demand_share_csv=analyze_dir / "high_demand_share_by_hour.csv",
    output_dir=output_dir,
    close_figs=False  # Keep figures open for display
)

# Display the figures
figs = report_out["figs"]
display(figs["fig1"])
display(figs["fig2"])

# Log figures to MLflow
with mlflow.start_run(run_name="Logging Artifacts", nested=True):
    mlflow.log_figure(figs["fig1"], "hourly_demand.png")
    mlflow.log_figure(figs["fig2"], "high_demand_share.png")
    print("Figures logged as artifacts.")

# Clean up
plt.close(figs["fig1"])
plt.close(figs["fig2"])


## 🤖 5. Automated Tracking with MLflow Autolog
MLflow can automatically log parameters and metrics for many libraries, including scikit-learn.


In [ ]:
from sklearn.linear_model import LinearRegression

# Enable autolog
mlflow.sklearn.autolog()

# Sample data
X = np.array([[1, 1], [1, 2], [2, 2], [2, 3]])
y = np.dot(X, np.array([1, 2])) + 3

# Run a simple model
with mlflow.start_run(run_name="Autolog Run"):
    model = LinearRegression()
    model.fit(X, y)
    print("Model fitted and autologged.")


## 🔍 6. Querying Experiment Data
We can use the MLflow client to retrieve results from our runs.


In [ ]:
from mlflow.tracking import MlflowClient

client = MlflowClient()
experiment = client.get_experiment_by_name(experiment_name)
runs = client.search_runs(experiment.experiment_id)

print(f"Found {len(runs)} runs in experiment '{experiment_name}':")
for run in runs:
    print(f"- Run ID: {run.info.run_id}, Name: {run.data.tags.get('mlflow.runName')}, Status: {run.info.status}")
